Setup

In [1]:
import Comove

DEBUG 0
DEBUG 1


In [15]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia
from astroquery.simbad import Simbad
import numpy as np
import pandas as pd
import os
import shutil
import time

# --- Load targets ---
arr = np.genfromtxt(
    "Targets.csv",
    delimiter=",",
    dtype=str,
    encoding="utf-8"
)

"""
--Header to ids-- 
hostname = 1
gaia id  = 2
ra       = 6
dec      = 7
st_rv    = 21
st_e_rv  = 22
"""

Simbad.server = "simbad.cds.unistra.fr"

# ---------- helpers ----------
def safe_name(s: str) -> str:
    """Make a Windows-safe folder/name (no spaces/colons/slashes, etc.)."""
    bad = '<>:"/\\|?*'
    for ch in bad:
        s = s.replace(ch, "_")
    return s.strip().replace(" ", "_")

def rename_file(star_name, full_directory_location, filename, gaia_id):
    og_path = os.path.join(full_directory_location, filename)
    if str(gaia_id) in filename:
        new_filename = filename.replace(str(gaia_id), star_name)
        new_path = os.path.join(full_directory_location, new_filename)
        os.replace(og_path, new_path)

def rename_directory(star_name, directory_location, gaia_id):
    """
    Renames files inside directory by replacing gaia_id -> star_name,
    then renames the directory itself the same way.
    Returns the NEW directory path.
    """
    full_path = os.path.abspath(directory_location)

    # rename files inside
    for filename in os.listdir(full_path):
        rename_file(star_name, full_path, filename, str(gaia_id))

    # rename directory
    if str(gaia_id) in os.path.basename(full_path):
        new_path = os.path.join(os.path.dirname(full_path),
                                os.path.basename(full_path).replace(str(gaia_id), star_name))
        os.rename(full_path, new_path)
        return new_path

    return full_path

# ---------- run one target ----------
targetIndex = 507

host_label = safe_name(arr[targetIndex][1])          # hostname column
gaia_id    = arr[targetIndex][2]                    # gaia id column

targname = f"Gaia DR3 {gaia_id}"
rd = [arr[targetIndex][6], arr[targetIndex][7]]     # ra/dec columns (strings OK if Comove expects that)

radvel = float(arr[targetIndex][21])                # st_rv column

vlim = 4.0
srad = 15.0

# --- Clean up any previous output folder BEFORE running ---
old_folder_by_host = f"./{host_label}_friends"
old_folder_by_id   = f"./{gaia_id}_friends"

if os.path.exists(old_folder_by_host):
    print(f"Cleaning up existing folder: {old_folder_by_host}")
    shutil.rmtree(old_folder_by_host)
elif os.path.exists(old_folder_by_id):
    print(f"Cleaning up existing folder: {old_folder_by_id}")
    shutil.rmtree(old_folder_by_id)

# --- Timed run (run ONCE) ---
t0 = time.perf_counter()
try:
    output_location = Comove.findfriends(
        targname,
        radvel,
        velocity_limit=vlim,
        search_radius=srad,
        radec=rd,                 # you had rd defined; pass it explicitly
        output_directory=None,
        verbose=False,
        showplots=False
    )
finally:
    print(f"Runtime: {time.perf_counter() - t0:.2f} s")

print("Comove output folder:", output_location)

# --- Rename output folder and internal files to hostname label ---
# (Only if Comove returned a valid folder path)
if output_location and os.path.exists(output_location):
    new_location = rename_directory(host_label, output_location, gaia_id)
    print("Renamed folder to:", new_location)

Asking Gaia for precise coordinates
INFO: Query finished. [astroquery.utils.tap.core]
Querying Gaia for neighbors
Parallax cut:  0.5
INFO: Query finished. [astroquery.utils.tap.core]
DEBUG 2
Populating RV table
DEBUG 3
DEBUG 3
Number with RV outside/inside selection range:  3   0
(-0.22324147234540956, -0.2537796283438367, 0.0004614183688906928)
XYZ (pc)   :  -223.2 -253.8 0.5
UVW (km/s) :  -17.86 -52.2 -11.59
Convergent point:  <SkyCoord (ICRS): (ra, dec, distance) in (deg, deg, )
    (109.74346675, -39.21179501, 999999.9)>
Searching on neighbors in GALEX
Searching on neighbors in 2MASS
Searching on neighbors in WISE
C:\Users\dshin\ASTR502\ASTR502spring26\Comove.py:1167: UserWarning: You passed a edgecolor/edgecolors ('black') for an unfilled marker ('+').  Matplotlib is ignoring the edgecolor in favor of the facecolor.  This behavior may change in the future.
  ddd = ax1.scatter( [ spt[yy[x]] ] , [ W13[yy[x]] ] , \

C:\Users\dshin\ASTR502\ASTR502spring26\Comove.py:1167: UserWarning: 

In [1]:
run run_stars.py

DEBUG 0
DEBUG 1

[8] Gliese_12 | Gaia DR3 2768048564768256512 | RV=51.04111099
Asking Gaia for precise coordinates
INFO: Query finished. [astroquery.utils.tap.core]
C:\Users\dshin\anaconda3\envs\ASTR502\Lib\site-packages\astropy\units\quantity.py:648: RuntimeWarning: invalid value encountered in arcsin
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)

Querying Gaia for neighbors
Parallax cut:  8.219379738137908
Note, using all-sky search


KeyboardInterrupt: 

'19.09913444519040'